In [356]:
from functools import partial

import numpy as np
import sympy as sp
from IPython.display import Math, display
from sympy.stats import (Covariance, E, Expectation, Normal, P, Variance,
                         variance)

In [ ]:
def Var(x):
    return Variance(x).evaluate_integral().simplify()


def Cov(x, y):
    return Covariance(x, y).evaluate_integral().simplify()


def show_tex(x):
    display(Math(sp.latex(x)))

In [302]:
def de_power(expr, replace_dict=None):
    if replace_dict is None:
        replace_dict = {}
    if isinstance(expr, sp.Mul):
        factors = expr.as_ordered_factors()
        res_lst = []
        for factor in factors:
            res_lst.append(de_power(factor, replace_dict=replace_dict))
        return sp.Mul(*res_lst)
    elif isinstance(expr, sp.Add):
        args = expr.args
        res_lst = []
        for arg in args:
            res_lst.append(de_power(arg, replace_dict=replace_dict))
        return sp.Add(*res_lst)
    elif isinstance(expr, sp.Pow):
        base, exp = expr.args
        if base in replace_dict:
            if isinstance(exp, sp.Number):
                _float_exp = float(exp)
                if _float_exp in replace_dict[base]:
                    return replace_dict[base][_float_exp]
        return expr
    else:
        return expr

def power_as_int(expr):
    if isinstance(expr, sp.Mul):
        factors = expr.as_ordered_factors()
        res_lst = []
        for factor in factors:
            res_lst.append(power_as_int(factor))
        return sp.Mul(*res_lst)
    elif isinstance(expr, sp.Add):
        args = expr.args
        res_lst = []
        for arg in args:
            res_lst.append(power_as_int(arg))
        return sp.Add(*res_lst)
    elif isinstance(expr, sp.Pow):
        base, exp = expr.args
        if exp == int(exp):
            return base ** int(exp)
        else:
            return expr
    else:
        return expr

def my_E(expr, replace_dict):
    return E(de_power(expr.expand(), replace_dict))


def my_Var(expr, replace_dict):
    return my_E((expr - my_E(expr, replace_dict))**2, replace_dict)


def my_Cov(expr1, expr2, replace_dict):
    return my_E((expr1 - my_E(expr1, replace_dict)) *
                (expr2 - my_E(expr2, replace_dict)), replace_dict)

## S/N Case

$x_{err} = k x + b + d $, $d \sim N(0, \sigma^2)$

In [15]:
mu_err, mu_obs, C_obs_err = sp.symbols('mu_err, mu_obs, C_obs_err')
S_err, S_obs = sp.symbols('S_err, S_obs', positive=True)

k, b, mu_x, = sp.symbols('k, b, mu_x')
sigma, S_x = sp.symbols('sigma, S_x', positive=True)

x = Normal('x', mu_x, S_x)
d = Normal('d', 0, sigma)

x_err = k * x + b + d

x_err_smb = sp.symbols('x_err_smb', positive=True)
e = Normal('e', 0, x_err_smb)

x_obs = x + e

In [17]:
eq1 = sp.Eq(E(x_err), mu_err)
show_tex(eq1)
eq2 = sp.Eq(E(x_obs), mu_obs)
show_tex(eq2)
eq3 = sp.Eq(Var(x_err), S_err**2)
show_tex(eq3)
eq4 = sp.Eq(E(Var(x_obs).subs(x_err_smb, x_err)).simplify(), S_obs**2)
show_tex(eq4)
eq5 = sp.Eq(Cov(x_err, x_obs), C_obs_err)
show_tex(eq5)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [18]:
solution = sp.solve((eq1, eq2, eq3, eq4, eq5), (k, b, sigma, mu_x, S_x))

In [19]:
for sol, var in zip(solution[-1], (k, b, sigma, mu_x, S_x)):
    display(Math(f"{sp.latex(var)} = {sp.latex(sol)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [20]:
for sol, var in zip(solution[-1], (k, b, sigma, mu_x, S_x)):
    print(f"{sp.latex(var)} = {sp.latex(sol)}")

k = - \frac{C_{obs err}}{S_{err}^{2} - S_{obs}^{2} + \mu_{err}^{2}}
b = \frac{C_{obs err} \mu_{obs} + S_{err}^{2} \mu_{err} - S_{obs}^{2} \mu_{err} + \mu_{err}^{3}}{S_{err}^{2} - S_{obs}^{2} + \mu_{err}^{2}}
\sigma = \sqrt{\frac{C_{obs err}^{2} + S_{err}^{4} - S_{err}^{2} S_{obs}^{2} + S_{err}^{2} \mu_{err}^{2}}{S_{err}^{2} - S_{obs}^{2} + \mu_{err}^{2}}}
\mu_{x} = \mu_{obs}
S_{x} = \sqrt{- S_{err}^{2} + S_{obs}^{2} - \mu_{err}^{2}}


In [472]:
def solve_SNR(mu_err, mu_obs, S_err, S_obs, C_obs_err):
    k = -C_obs_err / (S_err**2 - S_obs**2 + mu_err**2)
    b = (C_obs_err * mu_obs + S_err**2 * mu_err - S_obs**2 * mu_err +
         mu_err**3) / (S_err**2 - S_obs**2 + mu_err**2)
    sigma = ((C_obs_err**2 + S_err**4 - S_err**2 * S_obs**2 +
              S_err**2 * mu_err**2) / (S_err**2 - S_obs**2 + mu_err**2))**0.5
    mu_x = mu_obs
    S_x = (-S_err**2 + S_obs**2 - mu_err**2)**0.5

    return k, b, sigma, mu_x, S_x

In [473]:


N = 100_000

x = np.random.uniform(1, 2, N)
x_err = 0.5 * x + 0.5 + np.random.normal(0, 0.2, N)
x_obs = x + x_err * np.random.normal(0, 1, N)

In [474]:
np.std(x)

0.2883279026054277

In [476]:
mu_err = np.mean(x_err)
mu_obs = np.mean(x_obs)
S_err = np.std(x_err)
S_obs = np.std(x_obs)
C_obs_err = np.cov(x_obs, x_err)[0, 1]

solve_SNR(mu_err, mu_obs, S_err, S_obs, C_obs_err) # k, b, sigma, mu_x, S_x

(0.5260375471592059,
 0.4617803093911931,
 0.19773653781000095,
 1.4983356793200768,
 0.27908256705622114)

## Possion Case

$x_{err} = k \sqrt{x} + b + d $, $d \sim N(0, \sigma^2)$

In [338]:
mu_err, mu_obs, C_obs_err, C_obs_err2, h_err = sp.symbols(
    'mu_err, mu_obs, C_obs_err, C_obs_err2, h_err')
S_err, S_obs = sp.symbols('S_err, S_obs', positive=True)

k, b, mu_x, mu_x_root = sp.symbols('k, b, mu_x, mu_x_root')
sigma, S_x, S_x_root = sp.symbols('sigma, S_x, S_x_root', positive=True)

mu_x_1p5, S_x_1p5 = sp.symbols('mu_x_1p5, S_x_1p5')

x = Normal('x', mu_x, S_x)
x_root = Normal('x_root', mu_x_root, S_x_root)
x_1p5 = Normal('x_1p5', mu_x_1p5, S_x_1p5)
d = Normal('d', 0, sigma)

x_err = k * x**0.5 + b + d

x_err_smb = sp.symbols('x_err_smb', positive=True)
e = Normal('e', 0, x_err_smb)

x_obs = x + e

In [280]:
de_power_dict = {x: {0.5: x_root, 1.5: x_1p5}}

_E = partial(my_E, replace_dict=de_power_dict)
_Var = partial(my_Var, replace_dict=de_power_dict)
_Cov = partial(my_Cov, replace_dict=de_power_dict)
_dp = partial(de_power, replace_dict=de_power_dict)

$\text{Var}(\sqrt{X})=E(X)-E(\sqrt{X})^2$

In [339]:
eq1 = sp.Eq(E(de_power(x_err, de_power_dict)), mu_err)
show_tex(eq1)

eq2 = sp.Eq(E(de_power(x_obs, de_power_dict)), mu_obs)
show_tex(eq2)

eq3 = sp.Eq(_Var(x_err).simplify(), S_err**2)
show_tex(eq3)

eq4 = sp.Eq(
    _E(_Var(x_obs).simplify().subs(x_err_smb, x_err)).simplify(), S_obs**2)
show_tex(eq4)

eq5 = sp.Eq(_Cov(x_obs, x_err), C_obs_err)
show_tex(eq5)

cov_in = (x_obs - _E(x_obs)) * (x_err - _E(x_err))**2
cov_in = cov_in.expand().subs({x_root: x**0.5, x_1p5: x**1.5})
cov_in = power_as_int(cov_in)
eq6 = sp.Eq(_E(cov_in).simplify(), C_obs_err2)
show_tex(eq6)


h3_in = (x_err - _E(x_err))**3
h3_in = h3_in.expand().subs({x_root: x**0.5, x_1p5: x**1.5})
h3_in = power_as_int(h3_in)
eq7 = sp.Eq(_E(h3_in).simplify(), h_err)
show_tex(eq7)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [351]:
for eq in (eq1, eq2, eq3, eq4, eq5, eq6, eq7):
    print(sp.latex(eq))

b + k \mu_{x root} = \mu_{err}
\mu_{x} = \mu_{obs}
k^{2} \mu_{x} - k^{2} \mu_{x root}^{2} + \sigma^{2} = S_{err}^{2}
S_{x}^{2} + b^{2} + 2 b k \mu_{x root} + k^{2} \mu_{x} + \sigma^{2} = S_{obs}^{2}
- k \mu_{x} \mu_{x root} + k \mu_{x 1p5} = C_{obs err}
k^{2} \left(S_{x}^{2} + 2 \mu_{x} \mu_{x root}^{2} - 2 \mu_{x 1p5} \mu_{x root}\right) = C_{obs err2}
k^{3} \left(- 3 \mu_{x} \mu_{x root} + \mu_{x 1p5} + 2 \mu_{x root}^{3}\right) = h_{err}


In [ ]:
# can not find analitical solution for this problem
solution = sp.solve((eq1, eq2, eq3, eq4, eq5, eq6, eq7),
                    (k, b, sigma, mu_x, S_x, mu_x_1p5, mu_x_root))

In [463]:
def calculate_momentum(k, b, sigma, mu_x, S_x, mu_x_1p5, mu_x_root):
    # Calculate the quantities based on the provided formulas
    mu_err = b + k * mu_x_root
    S_err_squared = k**2 * mu_x - k**2 * mu_x_root**2 + sigma**2
    S_obs_squared = S_x**2 + b**2 + 2 * b * k * mu_x_root + k**2 * mu_x + sigma**2
    C_obs_err = -k * mu_x * mu_x_root + k * mu_x_1p5
    C_obs_err2 = k**2 * (S_x**2 + 2 * mu_x * mu_x_root**2 -
                         2 * mu_x_1p5 * mu_x_root)
    h_err = k**3 * (-3 * mu_x * mu_x_root + mu_x_1p5 + 2 * mu_x_root**3)

    S_err = np.sqrt(np.clip(S_err_squared, 0, np.inf))
    S_obs = np.sqrt(np.clip(S_obs_squared, 0, np.inf))

    return mu_err, S_err, S_obs, C_obs_err, C_obs_err2, h_err

In [464]:
N = 100_000

x = np.random.uniform(1, 2, N)
x_err = 0.7 * np.sqrt(x) + 0.5 + np.random.normal(0, 0.2, N)
x_obs = x + x_err * np.random.normal(0, 1, N)

In [470]:
mu_err_cal, S_err_cal, S_obs_cal, C_obs_err_cal, C_obs_err2_cal, h_err_cal = calculate_momentum(
    0.7, 0.5, 0.2, 1.5, np.std(x), np.mean(x**1.5), np.mean(x**0.5))

print("mu_err_cal:", mu_err_cal)
print("S_err_cal:", S_err_cal)
print("S_obs_cal:", S_obs_cal)
print("C_obs_err_cal:", C_obs_err_cal)
print("C_obs_err2_cal:", C_obs_err2_cal)
print("h_err_cal:", h_err_cal)

mu_err_cal: 1.3529062168411312
S_err_cal: 0.21806188404613286
S_obs_cal: 1.400397486934588
C_obs_err_cal: 0.02291723950029123
C_obs_err2_cal: 0.001678871154803154
h_err_cal: -0.0016511172113703187


In [471]:
mu_err_sample = np.mean(x_err)
S_err_sample = np.std(x_err)
S_obs_sample = np.std(x_obs)
C_obs_err_sample = np.cov(x_obs, x_err)[0, 1]
C_obs_err2_sample = np.mean(
    (x_obs - np.mean(x_obs)) * (x_err - np.mean(x_err))**2)
h_err_sample = np.mean((x_err - np.mean(x_err))**3)

print("mu_err:", mu_err_sample)
print("S_err:", S_err_sample)
print("S_obs:", S_obs_sample)
print("C_obs_err:", C_obs_err_sample)
print("C_obs_err2:", C_obs_err2_sample)
print("h_err:", h_err_sample)

mu_err: 1.3521607073336976
S_err: 0.2161338677533768
S_obs: 1.3959844967218282
C_obs_err: 0.022867869398520885
C_obs_err2: -5.7856542200803694e-05
h_err: -7.710662377980143e-06


In [456]:
from scipy.optimize import minimize, differential_evolution


def objective_function(params, mu_err, S_err, S_obs, C_obs_err, C_obs_err2,
                       h_err):
    k, b, sigma, mu_x, S_x, mu_x_1p5, mu_x_root = params
    mu_err_pred, S_err_pred, S_obs_pred, C_obs_err_pred, C_obs_err2_pred, h_err_pred = calculate_quantities(
        k, b, sigma, mu_x, S_x, mu_x_1p5, mu_x_root)

    # Objective: Minimize the sum of squares of the calculated quantities
    return np.sum([(mu_err - mu_err_pred)**2, (S_err - S_err_pred)**2,
                   (S_obs - S_obs_pred)**2, (C_obs_err - C_obs_err_pred)**2,
                   (C_obs_err2 - C_obs_err2_pred)**2, (h_err - h_err_pred)**2])


# Initial guess for the parameters
initial_guess = [
    1.0, 0.0, 0.2, 1.5,
    np.std(x), np.mean(x**1.5),
    np.mean(x**0.5)
]

# Bounds for the parameters (optional)
bounds = [(0, 2), (0, 2), (0, 2), (0, 2), (0, 2), (0, 2),
          (0, 2)]  # No bounds, but you can set them if needed

# Perform the optimization
result = differential_evolution(objective_function,
                                bounds,
                                args=(mu_err, S_err, S_obs, C_obs_err,
                                      C_obs_err2, h_err),
                                maxiter=10000)

In [460]:
print('Truth:', 0.7, 0.5, 0.2, 1.5, np.std(x), np.mean(x**1.5), np.mean(x**0.5))
print('Guess:', *result.x)

Truth: 0.7 0.5 0.2 1.5 0.2884112970654822 1.862214519155315 1.2188541391539454
Guess: 0.0457464109830098 1.3426231436543912 0.20700897589521938 1.9627747793416386 0.2921628366136442 1.032353271491246 0.256350386119365
